# 02 - Predicting Bid Success (Win_Loss)

**Goal:** estimate the probability that a bid will be won, using only information known at bid time.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src import config, quality, integrate
from src.load import load_all
from src.clean import clean_all
sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
FIG = config.FIGURES_DIR
def save(fig, name): fig.tight_layout(); fig.savefig(FIG / name, dpi=150)
from src.features import get_xy, NUMERIC_FEATURES, CATEGORICAL_FEATURES, EXCLUDED
from src import train

## 1. Data preparation
Input is the cleaned, integrated **Bid 360** table (one row per project).

In [ ]:
clean, _ = clean_all(load_all())
bid = integrate.build_bid_360(clean)
X, y = get_xy(bid)
print(X.shape, "| win rate:", round(y.mean(), 3))
print("Numeric:", NUMERIC_FEATURES); print("Categorical:", CATEGORICAL_FEATURES)
pd.Series(EXCLUDED, name="why excluded")

**Engineered features**
- `bid_to_cost_ratio` - implied margin (bid / total estimated cost)
- `bid_per_mw` - price competitiveness
- `share_*` - cost structure (share of each cost component)
- `handover_year`, `has_handover_date`

**Leakage check:** `handover_date` is filled for 36 % of won and 36 % of lost bids -> it does not reveal the outcome.

In [ ]:
bid.groupby(bid.handover_date.notna()).win_loss.agg(["mean", "count"])

**Preprocessing (inside the pipeline, fitted on training data only):**
numbers -> median fill + missing indicator + scaling; categories -> 'Unknown' + one-hot encoding.

## 2. Models and evaluation
80/20 stratified split. 5-fold cross-validation on the 80 % to compare models; the 20 % is used once at the end.

In [ ]:
res = train.run(X, y)
res["cv"][["model", "roc_auc", "roc_auc_std", "accuracy", "precision", "recall", "f1"]]

In [ ]:
pd.DataFrame(res["test"]).T

## 3. Results

In [ ]:
from IPython.display import Image, display
for f in ["ml_confusion_matrix.png", "ml_feature_importance.png"]:
    display(Image(filename=str(FIG / f), width=520))

## 4. Interpretation - Logistic Regression coefficients
Positive = increases win probability.

In [ ]:
pipe = __import__("joblib").load(config.PROCESSED_DIR / "win_model.joblib")
names = pipe.named_steps["prep"].get_feature_names_out()
coef = pd.Series(pipe.named_steps["model"].coef_[0], index=names).sort_values()
pd.concat([coef.head(8), coef.tail(8)]).round(2)

## 5. Conclusions
- **Logistic Regression** is selected: best cross-validated ROC-AUC, most stable, and fully explainable.
- The more complex tree models did **not** beat it -> relationships in this data are mostly simple/linear.
- Main drivers: plant efficiency, customer, country/region, plant size and bid value.
- **Limitations:** ~1,900 bids; no competitor, pricing-strategy or timeline data; label quality depends on Salesforce/MG updates (C3).
- **Use:** a win-probability score to prioritise bid effort - not an automatic decision.